# 04 - Models

Purpose: run full SOH and RUL model benchmarks with leakage-safe battery holdout evaluation.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Train benchmark model suites

In [2]:
from src.models import run_soh_benchmark, run_rul_benchmark
from src.evaluation import save_metrics

feature_df = pd.read_csv(PROCESSED_DIR / 'features_all.csv')
soh_metrics, soh_results = run_soh_benchmark(feature_df=feature_df, models_dir=MODELS_DIR)
rul_metrics, rul_results = run_rul_benchmark(feature_df=feature_df, models_dir=MODELS_DIR, include_transformer=True)
metrics_df = save_metrics(soh_metrics + rul_metrics, METRICS_PATH)
metrics_df

,Model,Task,RMSE,MAE,MAPE,R2,Train_Time_s
0,Random Forest Regressor,RUL,6.373208,5.091485,17.973037,0.962434,0.256819
1,XGBoost Regressor,RUL,6.385948,4.935395,15.623504,0.962284,0.919269
2,Transformer Encoder,RUL,6.583761,5.866489,50.553942,0.951852,5.994190
3,LSTM Neural Network,RUL,33.463212,29.424875,198.333743,-0.243854,1.230259
4,Linear Regression,SOH,0.116027,0.099809,0.127294,0.999769,0.000444
5,XGBoost Regressor,SOH,0.565523,0.367674,0.472972,0.994523,4.504291
6,Random Forest Regressor,SOH,0.714788,0.408414,0.530208,0.991250,0.187762
7,Ridge Regression,SOH,0.943786,0.885303,1.143149,0.984746,0.000350
8,CNN-BiLSTM,SOH,6.894650,6.183861,8.181457,0.013460,0.897369
9,LSTM Neural Network,SOH,6.941440,6.231954,8.022731,0.000024,1.134324


## Save core comparison plots

In [3]:
import numpy as np
from src.visualisation import plot_actual_vs_predicted, plot_rul_error, plot_rmse_comparison

for model_name, payload in soh_results.items():
    safe = model_name.lower().replace(' ', '_').replace('-', '_')
    plot_actual_vs_predicted(np.asarray(payload['cycle_number']), np.asarray(payload['y_true']), np.asarray(payload['predictions']),
                             f'SOH: Actual vs Predicted ({model_name}) on B0018', 'SOH (%)', FIG_DIR / f'soh_actual_vs_pred_{safe}.png')

for model_name, payload in rul_results.items():
    safe = model_name.lower().replace(' ', '_').replace('-', '_')
    plot_actual_vs_predicted(np.asarray(payload['cycle_number']), np.asarray(payload['y_true']), np.asarray(payload['predictions']),
                             f'RUL: Actual vs Predicted ({model_name}) on B0018', 'RUL (cycles)', FIG_DIR / f'rul_actual_vs_pred_{safe}.png')
    plot_rul_error(np.asarray(payload['cycle_number']), np.asarray(payload['y_true']), np.asarray(payload['predictions']), FIG_DIR / f'rul_error_{safe}.png')

plot_rmse_comparison(metrics_df, FIG_DIR / '08_rmse_comparison_soh_rul.png')
print('Model outputs generated.')

Model outputs generated.
